# Deep Reinforcement Learning — Doom Agent (SS2025)

Welcome to the last assignment for the **Deep Reinforcement Learning** course (SS2025). In this notebook, you'll implement and train a reinforcement learning agent to play **Doom**.

You will:
- Set up a custom VizDoom environment with shaped rewards
- Train an agent using an approach of your choice
- Track reward components across episodes
- Evaluate the best model
- Visualize performance with replays and GIFs
- Export the trained agent to ONNX to submit to the evaluation server

In [37]:
IN_COLAB = False
USE_GOOGLE_DRIVE = False

# -------- OS / Colab detection --------
import os
import sys
import shutil
import platform
import subprocess

OS_NAME = platform.system()   # "Windows", "Linux", "Darwin"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

print("OS:", OS_NAME, "| Colab:", IN_COLAB, "| Google Drive:", USE_GOOGLE_DRIVE)

if IN_COLAB:
  # -------- Install dependencies --------
  !apt update
  !apt install -y swig python3-numpy python3-dev cmake zlib1g-dev libjpeg-dev xvfb ffmpeg xorg-dev python3-opengl libboost-all-dev libsdl2-dev
  !pip install torch==2.10 numpy==2.4.2 matplotlib vizdoom==1.3.0 portpicker==1.6.0 gymnasium==1.2.3 onnx==1.21.0 onnx2pytorch==0.5.3 onnxscript==0.7.0 einops==0.8.2
  # -------- Clone repository --------
  !git clone https://$token@github.com/gerkone/jku.wad.git
  %cd jku.wad
  
  if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# -------- Device selection --------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

OS: Darwin | Colab: False | Google Drive: False


In [38]:
#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
from typing import Dict, Sequence

import torch
import wandb
from torch.distributions.categorical import Categorical

from collections import deque, OrderedDict
from copy import deepcopy
import random
import numpy as np
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from matplotlib import pyplot as plt
from PIL import Image
import vizdoom as vzd
from vizdoom import ScreenFormat

from gymnasium import Env
from torch import nn
from einops import rearrange

from agents.dqn import epsilon_greedy, update_ema
from doom_arena import VizdoomMPEnv
from doom_arena.reward import VizDoomReward
from doom_arena.render import render_episode
from IPython.display import HTML
from typing import Dict, Tuple
from doom_arena.reward import VizDoomReward


DTYPE = torch.float32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: " + str(device))
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Device: cpu


False

## Environment configuration

ViZDoom supports multiple visual buffers that can be used as input for training agents. Each buffer provides different information about the game environment, as seen from left to right:


Screen
- The default first-person RGB view seen by the agent.

Labels
- A semantic map where each pixel is tagged with an object ID (e.g., enemy, item, wall).

Depth
- A grayscale map showing the distance from the agent to surfaces in the scene.

Automap
- A top-down schematic view of the map, useful for global navigation tasks.

![buffers gif](https://vizdoom.farama.org/_images/vizdoom-demo.gif)

In [39]:
USE_GRAYSCALE = True  # ← flip to False for RGB

PLAYER_CONFIG = {
    # NOTE: "algo_type" defaults to POLICY in evaluation script!
    "algo_type": "QVALUE",  # OPTIONAL, change to POLICY if using policy-based (eg PPO)
    "n_stack_frames": 4,
    "extra_state": ["depth"],
    "hud": "none",
    "crosshair": True,
    "screen_format": 8 if USE_GRAYSCALE else 0,
}

In [56]:
# TODO: environment training paramters
N_STACK_FRAMES = PLAYER_CONFIG["n_stack_frames"] #1
NUM_BOTS = 4 # 4
EPISODE_TIMEOUT = 2000 # At test time 2000
# TODO: model hyperparams
GAMMA = 0.99 #0.95
EPISODES = 500 #100
BATCH_SIZE = 64 #32
REPLAY_BUFFER_SIZE = 40_000 #10_000
LEARNING_RATE = 2.5e-4 #1e-4
EPSILON_START = 1.0
EPSILON_END = 0.1 #0.1
EPSILON_DECAY = 0.995 #0.99
N_EPOCHS = 10 #10

wandb_config = {
    "gamma": GAMMA,
    "episodes": EPISODES,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epsilon_decay": EPSILON_DECAY,
    "n_epochs": N_EPOCHS,
    "replay_buffer_size": REPLAY_BUFFER_SIZE,
    "num_bots": NUM_BOTS,
    "stack_frames": N_STACK_FRAMES
}

run = wandb.init(
    project="RL_DOOM", 
    config=wandb_config
)

## Reward function
In this task, you will define a reward function to guide the agent's learning. The function is called at every step and receives the current and previous game variables (e.g., number of frags, hits taken, health).

Your goal is to combine these into a meaningful reward, encouraging desirable behavior, such as:

- Rewarding frags (enemy kills)

- Rewarding accuracy (hitting enemies)

- Penalizing damage taken

- (Optional) Encouraging survival, (ammo efficiency not used), etc.

You can return multiple reward components, which are summed during training. Consider the class below as a great starting point!

In [57]:
class YourReward(VizDoomReward):
    def __init__(self, num_players: int):
        super().__init__(num_players)

    def __call__(
        self,
        vizdoom_reward: float,
        game_var: Dict[str, float],
        game_var_old: Dict[str, float],
        player_id: int,
    ) -> Tuple[float, float, float]:
        """
        Custom reward function used by both training and evaluation.
        *  +100  for every new frag
        *  +2    for every hit landed
        *  -0.1  for every hit taken
        """
        self._step += 1
        _ = vizdoom_reward, player_id  # unused

        rwd_hit = 2.0 * (game_var["HITCOUNT"] - game_var_old["HITCOUNT"])
        rwd_hit_taken = -0.1 * (game_var["HITS_TAKEN"] - game_var_old["HITS_TAKEN"])
        rwd_frag = 100.0 * (game_var["FRAGCOUNT"] - game_var_old["FRAGCOUNT"])

        
        # 2. Movement & Anti-Corner / Anti-Stuck Mechanics
        # Calculate distance traveled in the last frame step
        dx = game_var.get("POSITION_X", 0.0) - game_var_old.get("POSITION_X", 0.0)
        dy = game_var.get("POSITION_Y", 0.0) - game_var_old.get("POSITION_Y", 0.0)
        distance_moved = math.sqrt(dx**2 + dy**2)

        # Exploration Reward: Reward the agent for actively moving through space
        # Gives a small positive reinforcement proportional to movement
        rwd_exploration = 0.01 * distance_moved

        # Anti-Corner Penalty: If the agent is stagnant (distance < thresh) but 
        # its internal step continues, it might be running into a wall or corner.
        rwd_stuck_penalty = 0.0
        if distance_moved < 0.5: 
            rwd_stuck_penalty = -0.01

        
        return rwd_hit, rwd_hit_taken, rwd_frag, rwd_exploration, rwd_stuck_penalty





In [58]:
reward_fn = YourReward(num_players=1)

env = VizdoomMPEnv(
    num_players=1,
    num_bots=NUM_BOTS,
    bot_skill=0,
    doom_map="ROOM",  # NOTE simple, small map; other options: TRNM, TRNMBIG
    extra_state=PLAYER_CONFIG[
        "extra_state"
    ],  # see info about states at the beginning of 'Environment configuration' above
    episode_timeout=EPISODE_TIMEOUT,
    n_stack_frames=PLAYER_CONFIG["n_stack_frames"],
    crosshair=PLAYER_CONFIG["crosshair"],
    hud=PLAYER_CONFIG["hud"],
    screen_format=PLAYER_CONFIG["screen_format"],
    reward_fn=reward_fn,
)

Host 53172
Player 53172


## Server Evaluation

Helper function to evaluate agent in the similar style as on the challenge server.

In [59]:
# ================================================================
# Helper code to evaluate over 10 episodes
# ================================================================

def _first_player_obs(obs):
    if isinstance(obs, tuple) and len(obs) == 2 and isinstance(obs[1], dict):
        obs = obs[0]
    if isinstance(obs, (list, tuple)):
        obs = obs[0]
    return obs


def _scalar_reward(reward):
    if isinstance(reward, (list, tuple)):
        reward = reward[0]
    if isinstance(reward, torch.Tensor):
        reward = reward.detach().cpu().reshape(-1)[0].item()
    elif isinstance(reward, np.ndarray):
        reward = reward.reshape(-1)[0]
    return float(reward)


def _server_like_eval_env(seed: int) -> VizdoomMPEnv:
    return VizdoomMPEnv(
        num_players=1,
        num_bots=4,
        bot_skill=0,
        doom_map="ROOM",
        episode_timeout=2000,
        screen_format=PLAYER_CONFIG["screen_format"],
        n_stack_frames=PLAYER_CONFIG["n_stack_frames"],
        extra_state=PLAYER_CONFIG.get("extra_state"),
        hud=PLAYER_CONFIG["hud"],
        crosshair=PLAYER_CONFIG["crosshair"],
        seed=seed,
    )


def _select_eval_action(model: nn.Module, obs):
    obs_tensor = torch.as_tensor(obs, device=device, dtype=DTYPE).unsqueeze(0)
    if len(obs_tensor.shape) == 5: 
        obs_tensor = obs_tensor.flatten(1, 2)
        
    with torch.no_grad():
        logits = model(obs_tensor)
        if isinstance(logits, tuple):
            logits = logits[0]
        if PLAYER_CONFIG.get("algo_type", "POLICY") == "POLICY":
            action = Categorical(logits=logits).sample()
        else:
            action = logits.argmax(dim=-1)
    return int(action.cpu().numpy()[0])


def evaluate(model: nn.Module, env: VizdoomMPEnv, n_episodes: int = 10, seed: int | None = None) -> float:
    """
    Server-like evaluation over `n_episodes` fresh ROOM episodes.
    Returns the median reward and prints the server mean plus diagnostics.
    """
    _ = env  # kept for the original notebook call signature
    model.eval()
    rewards = []
    seed_rng = np.random.default_rng(seed)

    for episode in range(n_episodes):
        episode_seed = int(seed_rng.integers(0, 10_000_000))
        eval_env = _server_like_eval_env(seed=episode_seed)
        try:
            obs = _first_player_obs(eval_env.reset())
            done = False
            total_reward = 0.0

            while not done:
                action = _select_eval_action(model, obs)
                step_result = eval_env.step(action)

                if len(step_result) == 5:
                    obs, reward, terminated, truncated, _ = step_result
                    done = terminated or truncated
                else:
                    obs, reward, done, _ = step_result

                obs = _first_player_obs(obs)
                total_reward += _scalar_reward(reward)
        finally:
            eval_env.close()

        rewards.append(total_reward)
        print(f"Episode {episode + 1:02}/{n_episodes}: seed={episode_seed}, reward={total_reward:.2f}")

    mean_reward = np.mean(rewards)
    print("Average reward over {} episodes: {:.2f}".format(n_episodes, mean_reward))
    return mean_reward

## Agent

Implement **your own agent** in the code cell that follows.

* In `agents/dqn.py` and `agents/ppo.py` you’ll find very small **skeletons**—they compile but are meant only as reference or quick tests.  
  Feel free to open them, borrow ideas, extend them, or ignore them entirely.
* The notebook does **not** import those files automatically; whatever class you define in the next cell is the one that will be trained.
* You may keep the DQN interface, switch to PPO, or try something else.
* Tweak any hyper-parameters (`PLAYER_CONFIG`, ε-schedule, optimiser, etc.) and document what you tried.


In [60]:
# ================================================================
# DQN — design your network here
# ================================================================


class DQN(nn.Module):
    """
    Deep-Q Network template.

    Expected behaviour
    ------------------
    forward(frame)      # frame: (B, C, H, W)  →  Q-values: (B, num_actions)

    What to add / change
    --------------------
    • Replace the two `NotImplementedError` lines.
    • Build an encoder (Conv2d / Conv3d) + a head (MLP or duelling).
    • Feel free to use residual blocks from `agents/utils.py` or any design you like.
    """

    def __init__(self, input_dim: int, action_space: int, hidden: int = 128):
        super().__init__()
        self.algo_type = "QVALUE"  # ← change to "POLICY" if using policy-based (eg PPO)
        self.encoder = nn.Sequential(
            # Layer 1: Smooth initial look at the 128x128 image. 
            # Output size: (B, 32, 62, 62)
            nn.Conv2d(input_dim, 32, kernel_size=5, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # Layer 2: Extract deeper structures (edges, enemy outlines).
            # Output size: (B, 64, 30, 30)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            # Layer 3: Focus on fine patterns and spatial relationships.
            # Output size: (B, 128, 14, 14)
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=0),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # Layer 4: Final dense feature aggregation before the head.
            # Output size: (B, 128, 6, 6)
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=0),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.flatten = nn.Flatten()
        
        # Exact mathematical calculation for our 128x128 spatial pipeline:
        # 128 channels * 6 * 6 feature map size = 4,608 features
        conv_out_features = 6272#128 * 6 * 6

        # Robust Linear Head with Dropout to prevent overfitting to specific rooms
        self.head = nn.Sequential(
            nn.Linear(conv_out_features, hidden),
            nn.ReLU(),
            nn.Dropout(p=0.1), # Helps the agent generalize across different game states
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, action_space)
        )

        # -------- TODO: define your layers ------------------------
        # Example (very small) baseline — delete or improve:
        #
        # self.encoder = nn.Sequential(
        #     nn.Conv2d(input_dim, 32, 8, stride=4), nn.ReLU(),
        #     nn.Conv2d(32, 64, 4, stride=2),       nn.ReLU(),
        #     nn.Conv2d(64, 64, 3, stride=1),       nn.ReLU(),
        # )
        # self.head = nn.Sequential(
        #     nn.Flatten(),
        #     nn.Linear(64 * 12 * 12, hidden), nn.ReLU(),
        #     nn.Linear(hidden, action_space),
        # )
        #
        # -----------------------------------------------------------
        #raise NotImplementedError("Define DQN layers")

    def forward(self, frame: torch.Tensor) -> torch.Tensor:
        # -------- TODO: implement forward -------------------------
        #
        # x = self.encoder(frame)
        # x = self.head(x)
        # return x
        #
        # -----------------------------------------------------------
        #raise NotImplementedError("Implement forward pass") 
        # Ensure input values are scaled between 0 and 1 if your wrapper doesn't do it
        if frame.max() > 1.0:
            frame = frame / 255.0
            
        features = self.encoder(frame)
        flat_features = self.flatten(features)
        q_values = self.head(flat_features)
        return q_values



In [61]:
# ================================================================
# Initialise your network
# ================================================================

# main Q-network
in_channels = 8#env.observation_space.shape[0]  # 1 if grayscale, else 3/4
model = DQN(
    input_dim=in_channels,
    action_space=env.action_space.n,
    hidden=64,  # change or ignore
).to(device, dtype=DTYPE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Num of params {num_params}")

Num of params 650888


## Optional behaviour cloning warm start

We provide an optional behaviour-cloning dataset that contains all observation buffers: screen, labels, depth, and automap. You may use it to pre-train your policy before starting RL. The helper below builds a dataloader from episode `.npz` files once the dataset is available locally.

Choose `BC_TRAIN_BUFFERS` to decide which of the provided buffers your policy should receive. This choice should match your current `PLAYER_CONFIG`. For example, if your RL agent uses screen, labels, depth, and automap, use `["screen", "labels", "depth", "automap"]`; if it uses only screen and depth, use `["screen", "depth"]`.

*TIP: You'll likely run out of System RAM while loading the data on Colab. Do this step locally, as training is relatively fast even on CPU*


In [46]:
# ================================================================
# Optional BC dataset download/setup
# ================================================================
# Import gdown, and install it first if it is missing
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

# Use the current notebook folder as the base directory
BASE_DIR = os.getcwd()

# Download one file from Google Drive if it does not already exist
def download_file(file_id, output_name):
    output_path = os.path.join(BASE_DIR, output_name)

    if os.path.isfile(output_path):
        print(f"{output_name} already exists, skipping download.")
        return output_path

    gdown.download(id=file_id, output=output_path, quiet=False)
    return output_path

# Download a zip file and extract it if the expected folder does not already exist
def download_and_extract(file_id, zip_name, expected_dir):
    expected_path = os.path.join(BASE_DIR, expected_dir)

    if os.path.isdir(expected_path):
        print(f"{expected_dir} already exists, skipping download and extraction.")
        return

    zip_path = download_file(file_id, zip_name)

    print(f"Extracting {zip_name}...")
    shutil.unpack_archive(zip_path, BASE_DIR, "zip")
    print(f"Extracted {zip_name}")


# Download and extract the behavior cloning dataset (if not already done)
download_and_extract(
    "1z3iOKTHjzlg8V6WdZFBl8I5y-GCl_UBJ",
    "doom_bc_data_top10.zip",
    "doom_bc_data_top10",
)

Downloading...
From (original): https://drive.google.com/uc?id=1z3iOKTHjzlg8V6WdZFBl8I5y-GCl_UBJ
From (redirected): https://drive.google.com/uc?id=1z3iOKTHjzlg8V6WdZFBl8I5y-GCl_UBJ&confirm=t&uuid=aa92b0fd-9d72-4479-975b-67dc76310166
To: /Users/kseniiatulei/jku.wad/doom_bc_data_top10.zip
100%|████████████████████████████████████████| 626M/626M [03:18<00:00, 3.16MB/s]


Extracting doom_bc_data_top10.zip...
Extracted doom_bc_data_top10.zip


In [47]:
# ================================================================
# Optional BC dataloader
# ================================================================
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split

# Directory containing the provided ep_XXXX.npz behaviour-cloning files.
# Update this if the download/setup cell stores the data somewhere else.
BC_DATA_DIR = Path("doom_bc_data_top10")
BC_SAVED_BUFFERS = ["screen", "labels", "depth", "automap"]
BC_TRAIN_BUFFERS = ["screen", *PLAYER_CONFIG.get("extra_state", [])] 
BC_BATCH_SIZE = 128
BC_VAL_FRACTION = 0.1


def bc_channel_indices(saved_buffers, train_buffers):
    widths = {"screen": 1 if USE_GRAYSCALE else 3, "labels": 1, "depth": 1, "automap": 3}
    saved = {}
    offset = 0
    for name in saved_buffers:
        width = widths[name]
        saved[name] = list(range(offset, offset + width))
        offset += width

    idx = []
    for name in train_buffers:
        idx.extend(saved[name])
    return idx


class BCDataset(Dataset):
    def __init__(self, data_dir, saved_buffers, train_buffers):
        self.files = sorted(Path(data_dir).glob("ep_*.npz"))
        if not self.files:
            raise FileNotFoundError(f"No BC files found in {data_dir!s}")
        self.channel_idx = bc_channel_indices(saved_buffers, train_buffers)
        self.episodes = []
        self.index = []

        for ep_idx, file in enumerate(self.files):
            data = np.load(file)
            states = data["states"].astype(np.float32)
            actions = data["actions"].astype(np.int64)
            self.episodes.append((states, actions))
            self.index.extend((ep_idx, t) for t in range(len(actions)))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        ep_idx, t = self.index[idx]
        states, actions = self.episodes[ep_idx]
        state = torch.from_numpy(states[t, self.channel_idx])
        action = torch.tensor(actions[t], dtype=torch.long)
        return state, action


bc_dataset = BCDataset(BC_DATA_DIR, BC_SAVED_BUFFERS, BC_TRAIN_BUFFERS)
bc_val_size = max(1, int(len(bc_dataset) * BC_VAL_FRACTION))
bc_train_size = len(bc_dataset) - bc_val_size
bc_train_dataset, bc_val_dataset = random_split(
    bc_dataset,
    [bc_train_size, bc_val_size],
    generator=torch.Generator().manual_seed(42),
)
bc_train_loader = DataLoader(bc_train_dataset, batch_size=BC_BATCH_SIZE, shuffle=True)
bc_val_loader = DataLoader(bc_val_dataset, batch_size=BC_BATCH_SIZE, shuffle=False)

print(
    f"Loaded {len(bc_dataset)} BC samples from {len(bc_dataset.files)} episodes | "
    f"channels={bc_dataset.channel_idx} | train={bc_train_size} val={bc_val_size}"
)


Loaded 20000 BC samples from 10 episodes | channels=[0, 2] | train=18000 val=2000


In [50]:
# ================================================================
# Optional BC training/evaluation warm start
# ================================================================
# TODO: implement behaviour cloning here before the RL loop.
#
# Suggested structure:
#   1. Train `model` on `bc_train_loader` with supervised action loss.
#      - Q-value models: logits_or_q = model(states); loss = F.cross_entropy(logits_or_q, actions)
#      - Policy models: use the policy logits and cross-entropy / negative log likelihood.
#   2. Evaluate action accuracy and/or loss on `bc_val_loader`.
#   3. Optionally copy the warm-started weights into `model_tgt`.
#   4. Continue with the RL training loop below.

# Example placeholders students may fill in:
# BC_EPOCHS = 5
# bc_optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

import os
import torch.nn.functional as F

# Define a clean filename for your pre-trained weights
BC_WEIGHTS_PATH = "bc.pt"

# 1. Configuration parameters
BC_EPOCHS = 5
bc_optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"Starting Behavioral Cloning Pre-training for {BC_EPOCHS} epochs...")

for bc_epoch in range(BC_EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    # --- Training Loop ---
    for states, actions in bc_train_loader:
        states = states.to(device, dtype=DTYPE)
        actions = actions.to(device, dtype=torch.long)
        
        ## Flatten channels exactly like our RL loop: [B, 2, 4, 128, 128] -> [B, 8, 128, 128]
        if states.shape[1] == 2:
            states = states.repeat(1, 4, 1, 1)
        #if len(states.shape) == 5:
            #states = states.flatten(1, 2)
            
        # Calculate predicted action values (Logits / Q-values)
        logits_or_q = model(states)
        
        # Supervised Loss: Cross Entropy forces the highest Q-value onto the target action
        loss = F.cross_entropy(logits_or_q, actions)
        
        bc_optimizer.zero_grad()
        loss.backward()
        bc_optimizer.step()
        
        # Tracking diagnostics
        train_loss += loss.item() * states.size(0)
        predictions = logits_or_q.argmax(dim=1)
        train_correct += (predictions == actions).sum().item()
        train_total += states.size(0)
        
    avg_train_loss = train_loss / train_total
    train_accuracy = (train_correct / train_total) * 100.0
    
    # --- Validation Loop ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for states, actions in bc_val_loader:
            states = states.to(device, dtype=DTYPE)
            actions = actions.to(device, dtype=torch.long)
            
            #if len(states.shape) == 5:
                #states = states.flatten(1, 2)
            if states.shape[1] == 2:
                states = states.repeat(1, 4, 1, 1)
                
            logits_or_q = model(states)
            loss = F.cross_entropy(logits_or_q, actions)
            
            val_loss += loss.item() * states.size(0)
            predictions = logits_or_q.argmax(dim=1)
            val_correct += (predictions == actions).sum().item()
            val_total += states.size(0)
            
    avg_val_loss = val_loss / val_total
    val_accuracy = (val_correct / val_total) * 100.0
    
    print(f"BC Epoch {bc_epoch+1:02}/{BC_EPOCHS:02} | "
          f"Train Loss: {avg_train_loss:.4f} ({train_accuracy:.1f}%) | "
          f"Val Loss: {avg_val_loss:.4f} ({val_accuracy:.1f}%)")
# Save the state dictionary of your trained model


torch.save(model.state_dict(), BC_WEIGHTS_PATH)
print(f"--> Behavioral Cloning weights successfully saved to disk at: '{BC_WEIGHTS_PATH}'")
print("Behavioral Cloning Pre-training Complete!\n")

Starting Behavioral Cloning Pre-training for 5 epochs...
BC Epoch 01/05 | Train Loss: 1.2272 (51.5%) | Val Loss: 1.1430 (54.7%)
BC Epoch 02/05 | Train Loss: 1.1053 (57.0%) | Val Loss: 1.1110 (57.1%)
BC Epoch 03/05 | Train Loss: 1.0236 (60.6%) | Val Loss: 1.0741 (57.4%)
BC Epoch 04/05 | Train Loss: 0.9453 (63.7%) | Val Loss: 1.0676 (56.2%)
BC Epoch 05/05 | Train Loss: 0.8790 (66.1%) | Val Loss: 1.0798 (56.6%)
--> Behavioral Cloning weights successfully saved to disk at: 'bc.pt'
Behavioral Cloning Pre-training Complete!



In [51]:
# ================================================================
# Optional BC evaluation before RL training
# ================================================================
eval_score = evaluate(model, env, n_episodes=10)

Host 53075
Player 53075
Episode 01/10: seed=4878471, reward=-1.00
Host 53076
Player 53076
Episode 02/10: seed=8030079, reward=0.00
Host 53077
Player 53077
Episode 03/10: seed=939449, reward=-2.40
Host 53078
Player 53078
Episode 04/10: seed=8288212, reward=-1.80
Host 53079
Player 53079
Episode 05/10: seed=4292335, reward=-0.10
Host 53082
Player 53082
Episode 06/10: seed=2247346, reward=-1.40
Host 53083
Player 53083
Episode 07/10: seed=1190523, reward=-2.40
Host 53084
Player 53084
Episode 08/10: seed=2788863, reward=-0.30
Host 53085
Player 53085
Episode 09/10: seed=3565388, reward=-2.20
Host 53086
Player 53086
Episode 10/10: seed=7155575, reward=-2.30
Average reward over 10 episodes: -1.39


## RL Training loop

In [62]:
# ================================================================
# Initialise your target network and training utilities
# ================================================================
import collections
import math
# TODO ------------------------------------------------------------
# 1. Create a target network (hard-copy or EMA)
# 2. Choose an optimiser + learning-rate schedule
# 3. Instantiate a replay buffer and set the initial epsilon value
#
# Hints:
#   model_tgt  = deepcopy(model).to(device)
#   optimiser  = torch.optim.Adam(...)
#   scheduler  = torch.optim.lr_scheduler.ExponentialLR(...)
#   replay_buf = collections.deque(maxlen=...)
# ---------------------------------------------------------------

#raise NotImplementedError("Create target net, optimiser, scheduler, replay buffer")

# 1. Create a target network identical to the main model
#model.load_state_dict(torch.load(BC_WEIGHTS_PATH, map_location=device))
model_tgt = deepcopy(model).to(device, dtype=DTYPE)
model_tgt.eval() # Target network is only used for evaluation steps

# 2. Choose an optimizer + learning-rate schedule
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)

# 3. Instantiate a replay buffer and set the initial epsilon value
replay_buffer = collections.deque(maxlen=REPLAY_BUFFER_SIZE)
epsilon = EPSILON_START

# Helper function for Target Network Exponential Moving Average (EMA) soft updates
def update_ema(target_net, main_net, tau=0.005):
    """
    Slowly blends main network weights into target network weights.
    Increases training stability significantly compared to sudden hard updates.
    """
    for target_param, main_param in zip(target_net.parameters(), main_net.parameters()):
        target_param.data.copy_(tau * main_param.data + (1.0 - tau) * target_param.data)

In [63]:
# ---------------------  TRAINING LOOP  ----------------------
# Feel free to change EVERYTHING below:
#   • choose your own reward function
#   • track different episode statistics in `ep_metrics`
#   • switch optimiser, scheduler, update rules, etc.
import random
reward_list, q_loss_list = [], []
best_eval_return, best_model = float("-inf"), None

#-----------------------new---------------------------------------------------
# Simple epsilon greedy implementation helper
def epsilon_greedy(env, policy_model, observation, eps, dev, dtype):
    if random.random() < eps:
        return env.action_space.sample()
    else:
        with torch.no_grad():
            # Add batch dimension to the input array shape
            obs_t = torch.tensor(observation, device=dev, dtype=dtype).unsqueeze(0)
            # If the single observation has that split dimension, flatten it too!
            if len(obs_t.shape) == 5: # [1, 2, 4, 128, 128]
                obs_t = obs_t.flatten(1, 2) # [1, 8, 128, 128]
            q_values = policy_model(obs_t)
            return q_values.argmax(dim=1).item()
#-----------------------new---------------------------------------------------



for episode in range(EPISODES):
    ep_metrics = {
        "custom_reward": 0.0,
        "hits_landed": 0,
        "frags": 0,
        "stuck_instances": 0
    }
    #ep_metrics = {"custom_reward": 0.0}  # ← add or replace keys as you like
    obs = env.reset()[0]
    done, ep_return = False, 0.0
    #model.eval()

    # ───────── rollout ─────────────────────────────────────────────
    while not done:
        model.eval()
        act = epsilon_greedy(env, model, obs, epsilon, device, DTYPE)
        next_obs, rwd_raw, done, _ = env.step(act)
        """
        # ----- reward definition (EDIT here) ----------------
        custom_rwd = float(rwd_raw[0])  # default: raw env reward
        # Example: access game variables for more detailed reward engineering
        # gv, gv_pre = env.envs[0].unwrapped._game_vars, env.envs[0].unwrapped._game_vars_pre
        # custom_rwd = your_function(gv, gv_pre)
        """
        #-----------------------new---------------------------------------------------

        # ----- Custom Reward Definition --------------------
        # Extract underlying game variables to calculate custom reward mechanisms
        try:
            gv = env.envs[0].unwrapped._game_vars
            gv_pre = env.envs[0].unwrapped._game_vars_pre
            
            # Pass our structured parameters directly through your custom reward class
            rwd_components = reward_fn(rwd_raw[0], gv, gv_pre, player_id=0)
            custom_rwd = sum(rwd_components)
            
            # Metric tracking diagnostics
            if gv["FRAGCOUNT"] > gv_pre["FRAGCOUNT"]: ep_metrics["frags"] += 1
            if gv["HITCOUNT"] > gv_pre["HITCOUNT"]: ep_metrics["hits_landed"] += 1
            if len(rwd_components) > 4 and rwd_components[4] < 0: ep_metrics["stuck_instances"] += 1
            
        except (AttributeError, IndexError):
            # Fallback if wrappers mask structural game metrics during certain frames
            custom_rwd = float(rwd_raw[0])
            
        # Append variables safely to our experience buffer
        # Storing observations as explicit torch tensors to preserve channel formatting
        obs_tensor = torch.tensor(obs, dtype=torch.float32)
        next_obs_tensor = torch.tensor(next_obs[0], dtype=torch.float32)
        
        replay_buffer.append((obs_tensor, act, custom_rwd, next_obs_tensor, done))
        
        obs = next_obs[0]
        ep_return += custom_rwd
    reward_list.append(ep_return)


        #-----------------------new---------------------------------------------------


        #ep_metrics["custom_reward"] += custom_rwd

        #replay_buffer.append((obs, act, custom_rwd, next_obs[0], done))
        #obs, ep_return = next_obs[0], ep_return + custom_rwd
    #reward_list.append(ep_return)

    # ───────── learning step (experience replay) ──────────────────
    current_loss = 0.0
    loss_count = 0
    if len(replay_buffer) >= BATCH_SIZE:
        model.train()
        for _ in range(N_EPOCHS):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            s, a, r, s2, d = zip(*batch)

            s = torch.stack(s).to(device, dtype=DTYPE)
            s2 = torch.stack(s2).to(device, dtype=DTYPE)
             
            if len(s.shape) == 5: # If shape is [64, 2, 4, 128, 128]
                s = s.flatten(1, 2)   # Becomes [64, 8, 128, 128]
                s2 = s2.flatten(1, 2) # Becomes [64, 8, 128, 128]
                
            a = torch.tensor(a, device=device, dtype=torch.long)
            r = torch.tensor(r, device=device, dtype=torch.float32)
            d = torch.tensor(d, device=device, dtype=torch.float32)

            q = model(s).gather(1, a.unsqueeze(1)).squeeze(1)
            """
            with torch.no_grad():
                q2 = model_tgt(s2).max(1).values
                tgt = r + GAMMA * q2 * (1 - d)
            loss = F.mse_loss(q, tgt)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            q_loss_list.append(loss.item())
            
            """
            #-----------------------new---------------------------------------------------
            # ───────────────────────────────────────────────────────────
            # DOUBLE DQN CRITICAL SWITCH
            # ───────────────────────────────────────────────────────────
            with torch.no_grad():
                # 1. Main model chooses the action indices maximizing future state values
                best_actions = model(s2).max(1)[1].unsqueeze(1)
                
                # 2. Target model evaluates the Q-value of those main model decisions
                q2 = model_tgt(s2).gather(1, best_actions).squeeze(1)
                
                # 3. Formulate unbiased target update equations
                tgt = r + GAMMA * q2 * (1.0 - d)
            
            loss = F.mse_loss(q, tgt)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            current_loss += loss.item() # Note: make sure this variable name matches what you wrote
            loss_count += 1
            q_loss_list.append(loss.item())
            #-----------------------new---------------------------------------------------
        update_ema(model_tgt, model, tau=0.005)

    scheduler.step()
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    
    # Calculate an average loss for this specific training sequence
    avg_loss = (current_loss / loss_count) if loss_count > 0 else 0.0

    
    #print(f"Ep {episode+1:03}: return {ep_return:6.1f}  |  ε {epsilon:.3f}")
    print(f"Ep {episode+1:03} | Return: {ep_return:6.1f} | ε: {epsilon:.3f} | "
          f"Frags: {ep_metrics['frags']} | Hits: {ep_metrics['hits_landed']} | "
          f"Stuck Penalties: {ep_metrics['stuck_instances']}")
    # ????????? quick evaluation for best-model tracking ???????????
    #eval_seed = int(eval_seed_rng.integers(0, 2**31 - 1))


    eval_seed = random.randint(0, 2**31 - 1)
    log_dict = {
        "episode/train_return": ep_return,
        "episode/epsilon": epsilon,
        "episode/frags": ep_metrics["frags"],
        "episode/hits_landed": ep_metrics["hits_landed"],
        "episode/stuck_count": ep_metrics["stuck_instances"],
        "train/loss": avg_loss,
        "train/lr": optimizer.param_groups[0]['lr']
    }

    # 2. Only evaluate and check for a new "best model" every 10 episodes
    if (episode + 1) % 10 == 0:
        print(f"\n--- Running Official Evaluation at Episode {episode+1} ---")
        eval_return = evaluate(model, env, n_episodes=5, seed=42)
        
        # Add the evaluation metric to the wandb log dictionary for this step
        log_dict["episode/official_eval_mean_return"] = eval_return
        
        if eval_return > best_eval_return:
            best_eval_return = eval_return
            best_model = deepcopy(model)
            print(f"--> New best model checkpoint reached! Saved Evaluation Return: {best_eval_return:.2f}\n")

    # 3. Log everything to wandb safely at the end of every episode
    wandb.log(log_dict)





    """
    #eval_return = evaluate(model, seed=eval_seed)
    if (episode + 1) % 10 == 0:
        eval_return = evaluate(model, env, n_episodes=5, seed=42)
        # Log evaluation progress directly to Weights & Biases
        wandb.log({"episode/official_eval_mean_return": eval_return}, step=episode)
        
    if eval_return > best_eval_return:
        best_eval_return = eval_return
        best_model = deepcopy(model)
        print(f"--> New best model checkpoint reached! Saved Evaluation Return: {best_eval_return:.2f}")
        #best_eval_return, best_model = eval_return, deepcopy(model)
    wandb.log({
        "episode/train_return": ep_return,
        "episode/eval_return": eval_return,
        "episode/epsilon": epsilon,
        "episode/frags": ep_metrics["frags"],
        "episode/hits_landed": ep_metrics["hits_landed"],
        "episode/stuck_count": ep_metrics["stuck_instances"],
        "train/loss": avg_loss,
        "train/lr": optimizer.param_groups[0]['lr']
    }, step=episode)
    """
    


# ---------------------  SAVE / EXPORT ---------------------------------------
final_model = best_model if best_model is not None else model  # choose best
wandb.finish()



/var/folders/rr/kg7mspmj7d95r4q493lmssq00000gn/T/ipykernel_98242/341437670.py:75: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  obs_tensor = torch.tensor(obs, dtype=torch.float32)
/var/folders/rr/kg7mspmj7d95r4q493lmssq00000gn/T/ipykernel_98242/341437670.py:76: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  next_obs_tensor = torch.tensor(next_obs[0], dtype=torch.float32)


Ep 001 | Return:    1.7 | ε: 0.995 | Frags: 0 | Hits: 0 | Stuck Penalties: 988


/var/folders/rr/kg7mspmj7d95r4q493lmssq00000gn/T/ipykernel_98242/341437670.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  obs_t = torch.tensor(observation, device=dev, dtype=dtype).unsqueeze(0)


Ep 002 | Return:   -0.4 | ε: 0.990 | Frags: 0 | Hits: 0 | Stuck Penalties: 1127
Ep 003 | Return:   -5.3 | ε: 0.985 | Frags: 0 | Hits: 0 | Stuck Penalties: 1227
Ep 004 | Return:   -0.1 | ε: 0.980 | Frags: 0 | Hits: 0 | Stuck Penalties: 1058
Ep 005 | Return:   -7.9 | ε: 0.975 | Frags: 0 | Hits: 0 | Stuck Penalties: 1466
Ep 006 | Return:   17.9 | ε: 0.970 | Frags: 0 | Hits: 2 | Stuck Penalties: 894
Ep 007 | Return:    1.6 | ε: 0.966 | Frags: 0 | Hits: 0 | Stuck Penalties: 1033
Ep 008 | Return:    6.4 | ε: 0.961 | Frags: 0 | Hits: 0 | Stuck Penalties: 940
Ep 009 | Return:    8.6 | ε: 0.956 | Frags: 0 | Hits: 2 | Stuck Penalties: 996
Ep 010 | Return:    0.2 | ε: 0.951 | Frags: 0 | Hits: 0 | Stuck Penalties: 1040

--- Running Official Evaluation at Episode 10 ---
Host 53176
Player 53176
Episode 01/5: seed=892509, reward=-0.10
Host 53177
Player 53177
Episode 02/5: seed=7739560, reward=-0.70
Host 53178
Player 53178
Episode 03/5: seed=6545715, reward=-0.40
Host 53179
Player 53179
Episode 04/5: 

KeyboardInterrupt: 

In [ ]:
# ================================================================
# Evaluate your final model here (before rendering episodes with `render_episode`).
# ================================================================
eval_score = evaluate(final_model, env, n_episodes=10)

## Dump to ONNX

In [ ]:
import onnx
import json


def onnx_dump(env, model, config, filename: str):
    # dummy state
    init_state = env.reset()[0].unsqueeze(0)

    # Export to ONNX
    torch.onnx.export(
        model.cpu(),
        args=init_state,
        f=filename,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        dynamo=False,
        external_data=False,    
    )
    onnx_model = onnx.load(filename)

    meta = onnx_model.metadata_props.add()
    meta.key = "config"
    meta.value = json.dumps(config)

    onnx.save(onnx_model, filename)


onnx_dump(env, final_model, PLAYER_CONFIG, filename="model.onnx")
print("Best network exported to model.onnx")

## Evaluation and Visualization

In this final section, you can evaluate your trained agent, inspect its performance visually, and analyze reward components over time.


In [ ]:
# ---------------------------------------------------------------
#  Reward-plot helper  (feel free to edit / extend)
# ---------------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt


def plot_reward_components(reward_log, smooth_window: int = 5):
    """
    Plot raw and smoothed episode-level reward components.

    Parameters
    ----------
    reward_log : list[dict]
        Append a dict for each episode, e.g. {"frag": …, "hit": …, "hittaken": …}
    smooth_window : int
        Rolling-mean window size for the smoothed curve.
    """
    if not reward_log:
        print("reward_log is empty – nothing to plot.")
        return

    df = pd.DataFrame(reward_log)
    df_smooth = df.rolling(window=smooth_window, min_periods=1).mean()

    # raw
    plt.figure(figsize=(12, 5))
    for col in df.columns:
        plt.plot(df.index, df[col], label=col)
    plt.title("Raw episode reward components")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # smoothed
    plt.figure(figsize=(12, 5))
    for col in df.columns:
        plt.plot(df.index, df_smooth[col], label=f"{col} (avg)")
    plt.title(f"Smoothed (window={smooth_window})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Hint for replay visualisation:
# ----------------------------------------------------------------
# env.enable_replay()
# ... run an evaluation episode ...
# env.disable_replay()
# replays = env.get_player_replays()
#
# from doom_arena.render import render_episode
# from IPython.display import HTML
# HTML(render_episode(replays, subsample=5).to_html5_video())
#
# Feel free to adapt or write your own GIF/MP4 export.